In [3]:
!pip install -U langchain langchain-community langchain-core langchain-google-genai

In [4]:
import os

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

In [5]:
# ----------------------------
# Configure Gemini API
# ----------------------------

# os.environ["GOOGLE_API_KEY"] = "YOUR_GOOGLE_API_KEY"
# genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
import os
from google.colab import userdata
from google import genai
# Load API key from Colab Secrets into environment variable
os.environ["GOOGLE_API_KEY"] = userdata.get("googleKey")

In [6]:
# ---------------------------
# LLM
# ---------------------------

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)


# ---------------------------
# Tool
# ---------------------------

@tool
def calculator(expression: str) -> str:
    """Evaluate a math expression."""
    return str(eval(expression))


tools = [calculator]


# ---------------------------
# Prompt
# ---------------------------

prompt = ChatPromptTemplate.from_messages(
[
("system", "You are a helpful assistant. Use the calculator tool when needed."),
("placeholder", "{chat_history}"),
("human", "{input}")
]
)


# ---------------------------
# Tool binding
# ---------------------------

llm_with_tools = llm.bind_tools(tools)

In [7]:



# ---------------------------
# Chain
# ---------------------------

chain = prompt | llm_with_tools


# ---------------------------
# Memory
# ---------------------------

store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]


agent = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history"
)



In [8]:
#
# ---------------------------
# Chat loop
# ---------------------------

while True:

    query = input("\nYou: ")

    if query.lower() == "exit":
        break

    response = agent.invoke(
        {"input": query},
        config={"configurable": {"session_id": "demo"}}
    )

    print("\nAgent:", response.content)


You: I am from MCA background

Agent: [{'type': 'text', 'text': "That's great! Do you have any questions or is there anything specific you'd like to discuss related to your MCA background?", 'extras': {'signature': 'Cp0CAb4+9vsiDKYxQRpmdNIZFrcRNwbaIUWCB6Z+83Aoa17lYjeBJSNkUGkyVakpQBR9JKNJGlYkNFYT44Ijk74VRJE53W9NgOYe+Oj+cfKFpwdG0arshgIgi3yk61kVAEfd0pQK3rG71wIu2lenX4yDC9ZaCJQlS9V7lmOxxpnBzv7/3dHrWmih0s3U19gFvm+c8+fmq61chHddL/nGnFKx3B3ajrKZLlrToob9sKa4nzcCQoK1UbRfuJRUnAUtf7VD3iV47VVHRdIpX1dFRTtRqFyD6xYJxlTab4cAVysh2KXXRgzQPRNQFiGS99AvlWt0BZPG9dnC2vxgbH6vIdoFP5Q2vSEfpNwUDAGCdAl5FGpeEdS7k07r4jtUsWbx'}}]

You: Which background i am from?

Agent: You just told me you are from an MCA background.

You: I need to go home, how can you help me?

Agent: [{'type': 'text', 'text': 'I can\'t physically help you go home, but I might be able to help in other ways. Could you tell me what kind of help you need? For example, are you looking for:\n\n*   **Directions?** (If so, where are you now and where is

KeyboardInterrupt: Interrupted by user